<p align="center">
<a href="https://duckietown.com"><img src="../assets/images/dtlogo.png" alt="Duckietown Logo" width="30%"></a>
</p>



# ROS 2 Nodes, Topics, and Interfaces

The previous notebook inspected the definitions of image and IMU messages. Those definitions describe how sensor data is represented, but displaying them does not establish that any data is being sent by a publisher node, or received by any subscriber node.

So, how do we check if there are publishers, topics, and messages are beingn received?

A small text publisher makes both observations possible without a sensor or a custom Python program. By the end of this notebook, the distinction between a **node**, a **topic**, and a **message** will be visible in command output. The same example will also show why communicating components need an agreed message type.

## Is anything publishing yet?

The commands in this notebook run in two terminals inside the dedicated environment described in the [README](../README.md#prerequisites). Terminal 1 will run the publisher; Terminal 2 will observe it.

Go ahead and open two terminals. In Terminal 1, load the ROS 2 environment:

```bash
source /opt/ros/${ROS2_DISTRO}/setup.bash
```

Successful sourcing normally produces no output and returns to the prompt. It makes the installed ROS 2 commands and libraries available in that terminal.

Before starting a publisher, inspect the visible nodes:

```bash
ros2 node list
```

In the initial verification run, this command is expected to print nothing and return to the prompt. That is a normal initial state when no application nodes have been started in the dedicated environment. An empty listing means no visible nodes were discovered by this command; it is not an installation error.

If other exercises are already running, their node names may appear. 

## Give the graph something to carry

A publisher needs a topic name, a message type, and values for the message fields. For example, a simple text message is sufficient for this investigation.

The familiar `ros2 interface show` command reveals the field that must be filled:

```bash
ros2 interface show std_msgs/msg/String
```

Omitting the introductory comments, this shows:

```text
string data
```

The message contains one field, `data`, holding text. The command-line tool can create a publisher and fill that field directly.

In Terminal 1, start a publisher:

```bash
ros2 topic pub --rate 1 --node-name example_publisher \
  /chatter std_msgs/msg/String "{data: '42'}"
```

Here, `--rate 1` establishes the publishing frequency in Hertz (Hz), i.e., one message per second, and `--node-name` gives the publishing node an identifiable name. `/chatter` is the topic, `std_msgs/msg/String` is its message type, and `"{data: '42'}"` assigns the text `42` to the message's `data` field.

Captured output begins:

```text
publisher: beginning loop
publishing #1: std_msgs.msg.String(data='42')

publishing #2: std_msgs.msg.String(data='42')
```

The numbering continues as messages are published. This command keeps running, so the terminal does not return to its prompt. Leave it running while the second terminal inspects the graph.

## Find the node and its topic

In Terminal 2, load the same ROS 2 environment:

```bash
source /opt/ros/${ROS2_DISTRO}/setup.bash
```

As before, successful sourcing normally produces no output. Now repeat the node listing:

```bash
ros2 node list
```

The previously empty listing should now contain the node created by the publishing command:

```text
/example_publisher
```

The next question is which topic carries its data. The `-t` option includes message types in the topic listing:

```bash
ros2 topic list -t
```

Captured output:

```text
/chatter [std_msgs/msg/String]
/parameter_events [rcl_interfaces/msg/ParameterEvent]
/rosout [rcl_interfaces/msg/Log]
```

The `/chatter` entry matches the topic and type selected in Terminal 1. The other two entries support parameter-change notifications and ROS logging; they are not additional sensor streams.

The distinction is now visible: `/example_publisher` names a participating node, while `/chatter` names a message stream. The node performs the publishing work; the topic identifies where subscribers can receive its messages.

<!--
Proposed checkpoint replacement
ID: node-definition
Source: #find-the-node-and-its-topic
Question: The node listing contains /example_publisher, and the topic listing contains /chatter. What does each name identify?
Model answer: /example_publisher identifies the named node created by the publishing command. /chatter identifies the topic on which that node publishes String messages. A node and its topic are different graph entities.
-->

## Check that a message actually arrives

So we have established a node that publshes data on a topic at a nominal frequency, and on a different terminal have discovered that node, and the topic it is connected to. A topic listing identifies an available interface, but it does not necessarily imply that there is data flowing through it. 

To "listen" to the data flowing inside topics, the `ros2 topic echo` command is used, which creates a temporary subscriber. The `--once` option requests one message and then exits. In Terminal 2, run: 

```bash
ros2 topic echo --once /chatter
```

You should see:

```text
data: '42'
---
```

The `data` field contains the text sent by Terminal 1. The `---` marks the end of the displayed message. After receiving it, the command returns to the prompt. To see in real time the messages flow, omit the `--once` parameter in `ros2 topic echo`.

With the publisher running, a message should arrive shortly. If the command remains silent, it is still waiting; that is different from the empty node listing before the publisher started. The first check is whether Terminal 1 is still publishing and both terminals were opened in the same exercise environment. `Ctrl-C` ends a waiting observation.

The publisher did not need to name this subscriber. It published on `/chatter`, and the temporary subscriber joined that topic. Other subscribers can join the same stream without changing the publisher.

## Why must the message types agree?

The displayed value resembles the number 42, but the published message contains text. Would a subscriber expecting an integer receive the same data?

The integer message definition makes the difference visible. Recalling:

```bash
ros2 interface show std_msgs/msg/Int32
```

We see (omitting introductory comments):

```text
int32 data
```

Both message definitions use the field name `data`, but one field contains text and the other contains a signed 32-bit integer. A subscriber expecting `Int32` does not receive the publisher's `String` messages merely because the topic and field names match. ROS 2 does not automatically convert the text into an integer.

The meaning of a value also belongs to the interface agreement. Two components exchanging a distance can agree on a numerical field type yet disagree about whether its value represents metres or centimetres. A usable interface defines both the structure and the meaning of its data.

Delivery policies must also be compatible. ROS 2 calls these __quality of service (QoS)__ settings. The defaults used by these commands support this exchange; later graph-inspection activities examine the policies needed by sensor streams.

<!--
Proposed checkpoint replacement
ID: message-contract 
Source: #why-must-the-message-types-agree
Question: A subscriber expects std_msgs/msg/Int32 on /chatter, while the running publisher sends std_msgs/msg/String with data containing '42'. Why is matching the topic name insufficient?
Model answer: The message types differ. The publisher sends text, while the subscriber expects a signed 32-bit integer. A shared topic name and field name do not provide automatic type conversion.
-->

## When a stream is not the whole interaction

The text exchange demonstrates the same communication pattern used for successive camera images: a publisher produces a stream, and subscribers receive its messages.

Other needs suggest different interfaces:

| Need | ROS 2 interface | What the interface provides |
|---|---|---|
| Receive successive images | Topic | A stream of messages |
| Request a component's current status | Service | A request and a response |
| Navigate to a goal while reporting progress | Action | A goal, feedback, a final result, and cancellation |
| Configure a node's controller gain | Parameter | A named configuration value owned by the node |

For example, a navigation task needs more than a stream of position readings. Another component must be able to submit a goal, follow progress, and request cancellation. An action expresses that interaction.

A parameter serves a different purpose: it configures a node's behavior. The node must expose and use the parameter; a Python constant does not automatically become one.

These examples illustrate interface choices. They do not imply that the corresponding navigation or configuration interfaces are running in the exercise environment.

<!--
Proposed checkpoint replacement
ID: interface-selection
Source: #when-a-stream-is-not-the-whole-interaction
Question: A navigation task must report progress and support cancellation before reaching its goal. Which interface best expresses those requirements?
Choices:
A. A topic carrying position readings.
B. A service returning one response when navigation finishes.
C. An action accepting a goal and providing feedback, a result, and cancellation.
Correct choice: C.
Model answer: An action represents the task and its progress, completion, and cancellation. A position topic supplies observations, while a service supplies a request-response exchange.
-->

## Finish the observation

Stop the publisher with `Ctrl-C` in Terminal 1. Its publication output stops and the shell prompt returns.

In Terminal 2, repeat:

```bash
ros2 node list
```

In the verification run, the listing became empty again after the publisher stopped. You can stop the publisher in Terminal 1 by selecting it and pressing `Ctrl-C`. Discovery can take a short time to reflect the change. If other application nodes are running, their names may remain.

The before-and-after observations connect the running program to its presence in the graph.

## Further reading

The ROS 2 Jazzy documentation describes [topics](https://docs.ros.org/en/jazzy/Concepts/Basic/About-Topics.html), [services](https://docs.ros.org/en/jazzy/Concepts/Basic/About-Services.html), [actions](https://docs.ros.org/en/jazzy/Concepts/Basic/About-Actions.html), and [parameters](https://docs.ros.org/en/jazzy/Concepts/Basic/About-Parameters.html).

## Checkpoint

The next cell contains the interactive self-check. A response must be entered or selected before the corresponding answer can be revealed.

## Summary and next steps

The investigation began with an installed message definition and no visible application nodes. Starting a publisher added a named node and a typed topic; receiving a message established delivery. Stopping the publisher removed that node from the graph.

These observations depended on both terminals discovering the same participants. [Notebook 4: ROS 2 Discovery, Domains, and Names](./4-ros-2-discovery-domains-and-names.ipynb) examines the settings that make that possible.


In [ ]:
import sys
from pathlib import Path

working_directory = Path.cwd()
parent_directory = working_directory.parent
if (parent_directory / "packages").is_dir():
    parent_directory_path = str(parent_directory)
    sys.path.insert(0, parent_directory_path)

from packages.checkpoint_self_check import display_checkpoint_self_checks

display_checkpoint_self_checks()
